In [13]:
# OCR: Licence Plate Recognition
# install tesseract.exe from https://github.com/UB-Mannheim/tesseract/wiki
# pip install pytesseract
# pip install ultralytics

from ultralytics import YOLO
import cv2
#import string
import pytesseract

# load models
#coco_model = YOLO('yolov8n.pt')
license_plate_detector = YOLO('license_plate_detector.pt')

# Path to the Tesseract executable
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

frame = cv2.imread('car2.png')

license_plates = license_plate_detector(frame)[0]

for license_plate in license_plates.boxes.data.tolist():
    x1, y1, x2, y2, score, class_id = license_plate
    #print(x1, y1, x2, y2)

    # crop license plate
    license_plate_crop = frame[int(y1):int(y2), int(x1): int(x2), :]

    # process license plate
    license_plate_crop_gray = cv2.cvtColor(license_plate_crop, cv2.COLOR_BGR2GRAY)
    _, license_plate_crop_thresh = cv2.threshold(license_plate_crop_gray, 64, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)

    # Recognize text from the license plate
    text = pytesseract.image_to_string(license_plate_crop_thresh, config='--psm 7') #--psm 11
    text = text.upper().replace(' ', '')
    print("Detected license plate Number is:", text.strip())

    # Draw a rectangle around the license plate
    cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
    cv2.putText(frame, text.strip(), (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36,255,12), 2)

# Display the result
frame = cv2.resize(frame, None, fx=0.5, fy=0.5)
cv2.imshow('License Plate Detection', frame)
cv2.waitKey(0)
cv2.destroyAllWindows()



0: 480x640 1 license_plate, 45.3ms
Speed: 3.9ms preprocess, 45.3ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)
Detected license plate Number is: AAF512|


In [5]:
# text detector based on CNN
# need to install opencv contrib module from source
# in Anaconda Powershell Prompt
# conda install pip
# pip uninstall opencv-python
# pip install opencv-contrib-python --user

import cv2
import matplotlib.pyplot as plt

img = cv2.imread('text_sample2.jpg')
det = cv2.text.TextDetectorCNN_create("textbox.prototxt", "TextBoxes_icdar13.caffemodel")

print(img.shape)
rects, probs = det.detect(img)

THR = 0.3
for i, r in enumerate(rects):
    if probs[i] > THR:
        cv2.rectangle(img, (r[0], r[1]), (r[0]+r[2], r[1]+r[3]), (0, 255, 0), 3)
                      
plt.figure(figsize=(10,8))
plt.axis('off')
plt.imshow(img[:,:,[2,1,0]])
plt.tight_layout()
plt.show()

AttributeError: module 'cv2' has no attribute 'text'

###### %%writefile test
## Practice：Licence Plate Detection
1. Input continuous images from 'car.mp4'.
2. For each frame, detect every car using YOLOv8 trained data 'yolov8n.pt'. (mark with red rectangles)
3. For each car, detect a licence plate using 'license_plate_detector.pt'. (mark with blue rectangle)
3. For each licence plate, OCR using Tesseract.
4. Print the recognized licence plate number above each detected licence plate. (putText() in green color)
5. Use whatever you learned this semester to improve the result.
6. Upload your .ipynb file.

In [19]:
#game_5: Licence Plate Recognition
# install tesseract.exe from https://github.com/UB-Mannheim/tesseract/wiki
# pip install pytesseract
# pip install ultralytics

from ultralytics import YOLO
import cv2
import pytesseract

# load models
license_plate_detector = YOLO('license_plate_detector.pt')

# Path to the Tesseract executable
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

scaling_factor = 0.3
cap = cv2.VideoCapture("car.mp4")
frame_seq = 0

while True:
    frame_seq += 1
    if frame_seq > 1700:
        frame_seq = 0
    cap.set(cv2.CAP_PROP_POS_FRAMES , frame_seq)
    status_cap, frame0 = cap.read()
    if not status_cap:
        break
    frame = cv2.resize(frame0, None, fx=scaling_factor, fy=scaling_factor)

    license_plates = license_plate_detector(frame)[0]

    for license_plate in license_plates.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = license_plate
        #print(x1, y1, x2, y2)
    
        # crop license plate
        license_plate_crop = frame[int(y1):int(y2), int(x1): int(x2), :]
    
        # process license plate
        license_plate_crop_gray = cv2.cvtColor(license_plate_crop, cv2.COLOR_BGR2GRAY)
        _, license_plate_crop_thresh = cv2.threshold(license_plate_crop_gray, 64, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    
        # Recognize text from the license plate
        text = pytesseract.image_to_string(license_plate_crop_thresh, config='--psm 7') #--psm 11
        text = text.upper().replace(' ', '')
        #print("Detected license plate Number is:", text.strip())
    
        # Draw a rectangle around the license plate
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
        cv2.putText(frame, text.strip(), (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36,255,12), 2)

    
    cv2.imshow("find_two_look_alike", frame)
    k = cv2.waitKey(1)
    if k == 27:
        break
        
cap.release()
cv2.destroyAllWindows()


0: 384x640 2 license_plates, 43.9ms
Speed: 2.9ms preprocess, 43.9ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 license_plates, 42.5ms
Speed: 1.8ms preprocess, 42.5ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 license_plates, 39.7ms
Speed: 2.3ms preprocess, 39.7ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 license_plates, 46.0ms
Speed: 1.8ms preprocess, 46.0ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 license_plates, 37.4ms
Speed: 2.4ms preprocess, 37.4ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 license_plates, 46.3ms
Speed: 2.2ms preprocess, 46.3ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 license_plates, 39.7ms
Speed: 3.0ms preprocess, 39.7ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 license_plates, 37.5ms
Speed: 2.1ms preprocess, 